# ♟️ Chess Numba — Google Colab 雲端搜尋調參執行器 (LMR Phase 2)

### 🏆 Phase 1 成果總結
* **Phase 1 冠軍基準**：六軸基礎 LMR 參數已全數定案寫入生產基準（`BASE 437 / HIST 204 / CUTNODE 1839 / TTCAP 1191 / MC 35 / TABLE 113%`），L2 實測 **+15.6 ± 16.9 Elo**！

### 🎯 Phase 2 核心調參六軸（戰術吃子、殺著反制與局勢動態）
1. `LMR_BAD_CAPTURE_BONUS`（當前基線 1046）：劣勢吃子（SEE < 0）額外減步懲罰。
2. `LMR_GOOD_CAPTURE_RELIEF`（當前基線 1030）：優勢吃子（SEE >= 0）減步寬減保護。
3. `LMR_KILLER_COUNTER_RELIEF`（當前基線 1024）：殺著步與對策步（Killer / Countermove）寬減保護。
4. `LMR_NO_TTMOVE_BONUS`（當前基線 1068）：局面無 TT Move 指引時的額外減步懲罰。
5. `LMR_TTMOVE_REDUCTION`（當前基線 1024）：走法命中 TT 走步時的基準減步。
6. `LMR_NOT_IMP_NUM`（當前基線 197）：靜態評估未改善（Not Improving）狀態下的動態減步係數。

### ⚡ 調參架構
* **L1 雲端 SPSA 探索**：80 輪 × 150 局 = 12,000 局 @ 50,000 nodes（inprocess 平行加速）
* **Polyak-Ruppert 平滑取樣**：嚴格鎖定最優經驗比例 **`--tail 0.3`**（後 24 輪 / 3,600 局平滑）
* **L2 深度獨立驗收**：600 局 @ 300,000 nodes，主開局庫 `data/openings.epd`，SPRT(0.0, 3.0)

In [ ]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version

In [ ]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout -f {branch}
# 清除前代暫存檔，確保 git pull 順暢
!rm -f tournament_analysis/l2_verify_lmr_p1_deep*.json tune_search/candidate_lmr_p1_deep*.json
!git pull origin {branch}
!pip install -q numba chess numpy

In [ ]:
# 3. 測試環境與驗證 Phase 2 基準常數
import sys
sys.path.insert(0, "/content/chess_numba")
from tune_search.search_param_registry import get_default_params

phase2_params = [
    "LMR_BAD_CAPTURE_BONUS",
    "LMR_GOOD_CAPTURE_RELIEF",
    "LMR_KILLER_COUNTER_RELIEF",
    "LMR_NO_TTMOVE_BONUS",
    "LMR_TTMOVE_REDUCTION",
    "LMR_NOT_IMP_NUM"
]
print("✅ 成功載入！LMR 第二階段核心六軸當前基準值（Baseline）：")
print(get_default_params(names=phase2_params))

In [ ]:
# 4. 啟動 L1 SPSA LMR 第二階段核心六軸雲端對抗調參（目標 80 輪 x 150 局 = 12,000 局）
import os
import json

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，調參並行度設為: {concurrency}")

log_file = "tune_search/logs/lmr_p2_deep_l1.jsonl"
state_file = "tune_search/states/lmr_p2_deep_l1.json"
total_campaign_iters = 80

# 讀取存檔狀態以計算剩餘需要執行的迭代輪數（支援斷點續跑）
if os.path.exists(state_file):
    with open(state_file, "r", encoding="utf-8") as f:
        saved_state = json.load(f)
    done_iters = saved_state.get("iteration", 0)
    remaining_iters = max(0, total_campaign_iters - done_iters)
    resume_flag = "--resume"
    print(f"🔄 斷點續跑模式：已完成 {done_iters}/{total_campaign_iters} 輪，本次接續執行剩餘 {remaining_iters} 輪！")
else:
    done_iters = 0
    remaining_iters = total_campaign_iters
    resume_flag = ""
    print(f"🚀 全新開跑模式：計劃執行全量 {remaining_iters} 輪！")

if remaining_iters > 0:
    !python -m tune_search.search_tuner \
      --params LMR_BAD_CAPTURE_BONUS,LMR_GOOD_CAPTURE_RELIEF,LMR_KILLER_COUNTER_RELIEF,LMR_NO_TTMOVE_BONUS,LMR_TTMOVE_REDUCTION,LMR_NOT_IMP_NUM \
      --backend inprocess \
      --scale-mode integer \
      --iter {remaining_iters} \
      --games 150 \
      --nodes 50000 \
      --concurrency {concurrency} \
      --adjudicate \
      --log {log_file} \
      --state {state_file} \
      {resume_flag}
else:
    print("✅ Phase 2 的 80 輪調參已全部完成！請直接執行步驟 5 產出候選參數。\n")

In [ ]:
# 5. 分析 SPSA 調參成果，以 Polyak-Ruppert 後 30% 平均（tail 0.3）產出最佳候選參數
!python tune_search/summarize_run.py tune_search/logs/lmr_p2_deep_l1.jsonl --tail 0.3 --out tune_search/candidate_lmr_p2_deep.json
print("\n=== Phase 2 最佳候選參數 JSON 內容 (tail 0.3) ===")
!cat tune_search/candidate_lmr_p2_deep.json

In [ ]:
# 6. 啟動 L2 深度 SPRT 實戰驗證（300,000 nodes, 600 局，主開局庫獨立驗證）
# 對照基準明確指定為原始 Baseline (tune_search/baseline_lmr_p2.json)
!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_lmr_p2_deep.json \
  --baseline-json tune_search/baseline_lmr_p2.json \
  --nodes 300000 \
  --games 600 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_lmr_p2_deep_tail03.json